# Task 20 - Text Vectorization (TF-IDF / Bag of Words) with Feature Selection

**Goal:** vectorize text with TF-IDF or Bag of Words, experiment with n-grams, stop-word removal and stemming / lemmatization, then optimise the feature set with chi-square or mutual-information scoring.

**Data mapping:** there is no review text in your data, so the text is the product **`title`** (short seller-written listing copy). Results describe which *words in a listing* separate higher- from lower-rated products.

**Target:** `high_rating` = rating above the median among products with at least `MIN_RATINGS` ratings (`TARGET_MODE = "sales"` switches to above-median `units_sold`).

**Experiments (all scored by 5-fold cross-validated ROC-AUC with the vectorizer and selector fitted inside each fold, so there is no selection leakage)**
1. Vectorizer: Bag of Words (counts) vs TF-IDF
2. n-grams: (1,1), (1,2), (1,3)
3. Stop words: none / English / English + data-driven domain words (appear in > 40% of titles)
4. Normalisation: none / stemming (Snowball) / lemmatization (WordNet, if available)
5. Selection on the best configuration: chi-square vs mutual information, k in {20, 50, 100, 200, 500, all}

**Input:** newest file before this task (missing `title` borrowed from raw)
**Outputs:** `data/processed/womens_clothing_tfidf_features.csv`, `reports/task_20_vectorizer_experiments.csv`, `reports/task_20_selected_features.csv`, `reports/task_20_analysis_status.csv`

In [ ]:
import re, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

def find_project_root():
    """Walk up from the notebook folder until a folder containing data/processed is found."""
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "data" / "processed").exists():
            return cand
    return start.parent

ROOT = find_project_root()
PROC, REPORTS = ROOT / "data" / "processed", ROOT / "reports"
RAW = ROOT / "data" / "raw" / "womens_clothing.csv"
PROC.mkdir(parents=True, exist_ok=True); REPORTS.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

def _norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

def pick(df, *candidates, required=True):
    """Return the real column name matching any candidate (ignores case, spaces, underscores)."""
    lookup = {_norm(c): c for c in df.columns}
    for cand in candidates:
        if _norm(cand) in lookup:
            return lookup[_norm(cand)]
    if required:
        raise KeyError(f"None of {candidates} found. Columns: {list(df.columns)}")
    return None

def load_first(*paths):
    """Load the first existing CSV, so each task builds on the newest previous output."""
    for p in paths:
        if Path(p).exists():
            d = pd.read_csv(p)
            d = d.loc[:, ~d.columns.str.startswith("Unnamed")]
            print(f"Loaded: {Path(p).name}  shape={d.shape}")
            return d, Path(p)
    raise FileNotFoundError("None of these files exist: " + ", ".join(str(p) for p in paths))

def add_from_raw(df, *names):
    """Borrow columns missing from df (e.g. title, tags) from data/raw, matched on product_id."""
    missing = [n for n in names if pick(df, n, required=False) is None]
    if not missing:
        return df
    raw = pd.read_csv(RAW); raw = raw.loc[:, ~raw.columns.str.startswith("Unnamed")]
    kd, kr = pick(df, "product_id", required=False), pick(raw, "product_id", required=False)
    if kd is None or kr is None:
        raise KeyError(f"Need product_id in both files to borrow {missing} from raw")
    got = [pick(raw, n) for n in missing if pick(raw, n, required=False)]
    if not got:
        raise KeyError(f"{missing} not found in raw file either")
    add = raw.drop_duplicates(kr)[[kr] + got].rename(columns={kr: kd})
    out = df.merge(add, on=kd, how="left")
    print("Borrowed from raw:", got)
    return out

# Newest cumulative dataset first; each task reads the newest file produced BEFORE it.
CHAIN = ["womens_clothing_high_cardinality_encoded.csv", "womens_clothing_engagement_features.csv",
         "womens_clothing_sentiment_features.csv", "womens_clothing_age_segmented.csv",
         "womens_clothing_review_text_features.csv", "womens_clothing_market_basket_features.csv",
         "womens_clothing_product_text_features.csv", "womens_clothing_product_tag_features.csv",
         "womens_clothing_product_segmented.csv", "womens_clothing_final.csv"]

# Tasks 19-21 outputs sit in front of the older chain.
LATE = ["womens_clothing_lifecycle_features.csv", "womens_clothing_tfidf_features.csv", "womens_clothing_channel_interaction_features.csv"]
FULL = LATE + CHAIN

import itertools
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
sns.set_theme(style="whitegrid"); pd.set_option("display.max_columns", 60)
RANDOM_STATE = 42
MIN_RATINGS = 20
TARGET_MODE = "rating"          # "rating" or "sales"
N_OUTPUT_TERMS = 30             # selected terms written to the dataset
MIN_DF = 3

## 1. Load data and build the target

In [ ]:
df, src = load_first(*[PROC / n for n in FULL[2:]])
df = add_from_raw(df, "title")
title_col = pick(df, "title"); rating_col = pick(df, "rating"); units_col = pick(df, "units_sold"); rc_col = pick(df, "rating_count", required=False)
df[title_col] = df[title_col].astype("string").fillna("").str.strip()
rel = (df[title_col].str.len() > 0) & ((df[rc_col] >= MIN_RATINGS) if rc_col else True)
if TARGET_MODE == "rating":
    y_all = (df[rating_col] > df.loc[rel, rating_col].median()).astype(int); target_name = "high_rating"
else:
    y_all = (df[units_col] > df[units_col].median()).astype(int); rel = df[title_col].str.len() > 0; target_name = "high_sales"
print(f"Target: {target_name} | products used: {rel.sum():,} | positive share: {y_all[rel].mean():.2f}")
texts_raw = df.loc[rel, title_col]; y = y_all[rel].values
texts_raw.head()

## 2. Text normalisation variants

All variants lower-case, strip digits and punctuation, and keep tokens of 2+ letters. Stemming uses the Snowball stemmer; lemmatization uses WordNet (downloaded once; skipped automatically if unavailable).

In [ ]:
from nltk.stem import SnowballStemmer
stemmer = SnowballStemmer("english")
TOKEN = re.compile(r"[a-z][a-z]+")
lemmatizer, lemma_ok = None, False
try:
    import nltk
    from nltk.stem import WordNetLemmatizer
    try: nltk.data.find("corpora/wordnet")
    except LookupError: nltk.download("wordnet", quiet=True)
    lemmatizer = WordNetLemmatizer(); lemmatizer.lemmatize("dresses"); lemma_ok = True
except Exception as e:
    print("Lemmatization unavailable ->", repr(e)[:120])

def tokens(t): return TOKEN.findall(str(t).lower())
norm_fns = {"none": lambda ws: ws, "stem": lambda ws: [stemmer.stem(w) for w in ws]}
if lemma_ok: norm_fns["lemma"] = lambda ws: [lemmatizer.lemmatize(w) for w in ws]
variants = {k: [" ".join(fn(tokens(t))) for t in texts_raw] for k, fn in norm_fns.items()}
print({k: v[0] for k, v in variants.items()})

# data-driven domain stop words: tokens present in > 40% of titles
doc_freq = pd.Series([w for t in variants["none"] for w in set(t.split())]).value_counts() / len(texts_raw)
domain_stop = sorted(doc_freq[doc_freq > 0.40].index)
print("Domain stop words:", domain_stop)
stop_sets = {"none": None, "english": list(ENGLISH_STOP_WORDS), "english+domain": list(ENGLISH_STOP_WORDS) + domain_stop}

## 3. Experiment grid (vectorizer x n-gram x stop words x normalisation)

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for vec_name, ngram, stop_name, norm_name in itertools.product(["bow", "tfidf"], [(1, 1), (1, 2), (1, 3)], stop_sets, norm_fns):
    # stop words are applied to the normalised text, so stem them too to keep the lists aligned
    sw = stop_sets[stop_name]
    if sw is not None and norm_name == "stem": sw = sorted({stemmer.stem(w) for w in sw})
    if sw is not None and norm_name == "lemma": sw = sorted({lemmatizer.lemmatize(w) for w in sw})
    V = CountVectorizer if vec_name == "bow" else TfidfVectorizer
    kw = dict(ngram_range=ngram, stop_words=sw, min_df=MIN_DF, token_pattern=r"(?u)\b[a-z][a-z]+\b")
    if vec_name == "bow": kw["binary"] = False
    pipe = Pipeline([("vec", V(**kw)), ("clf", LogisticRegression(max_iter=2000, C=0.5))])
    sc = cross_val_score(pipe, variants[norm_name], y, cv=cv, scoring="roc_auc")
    n_feat = len(pipe.named_steps["vec"].fit(variants[norm_name]).get_feature_names_out())
    rows.append({"vectorizer": vec_name, "ngram": f"{ngram[0]}-{ngram[1]}", "stop_words": stop_name, "normalisation": norm_name,
                 "n_features": n_feat, "cv_auc_mean": sc.mean(), "cv_auc_std": sc.std()})
exp = pd.DataFrame(rows).sort_values("cv_auc_mean", ascending=False).reset_index(drop=True)
exp.head(15).round(4)

In [ ]:
# Which choice matters most? Average AUC per option
fig, ax = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
for a_, c in zip(ax, ["vectorizer", "ngram", "stop_words", "normalisation"]):
    exp.groupby(c)["cv_auc_mean"].mean().plot.bar(ax=a_, color="#4c72b0"); a_.set_title(f"Mean CV AUC by {c}"); a_.set_ylim(exp["cv_auc_mean"].min() - .01, exp["cv_auc_mean"].max() + .01)
plt.tight_layout(); plt.show()
best_cfg = exp.iloc[0]; print("Best configuration:", dict(best_cfg[["vectorizer", "ngram", "stop_words", "normalisation"]]), "AUC", round(best_cfg["cv_auc_mean"], 4))

## 4. Feature selection: chi-square vs mutual information

In [ ]:
ng = tuple(int(i) for i in best_cfg["ngram"].split("-")); norm_name = best_cfg["normalisation"]
sw = stop_sets[best_cfg["stop_words"]]
if sw is not None and norm_name == "stem": sw = sorted({stemmer.stem(w) for w in sw})
if sw is not None and norm_name == "lemma": sw = sorted({lemmatizer.lemmatize(w) for w in sw})
V = CountVectorizer if best_cfg["vectorizer"] == "bow" else TfidfVectorizer
vec_kw = dict(ngram_range=ng, stop_words=sw, min_df=MIN_DF, token_pattern=r"(?u)\b[a-z][a-z]+\b")
total_features = best_cfg["n_features"]

def mi_score(X, yy): return mutual_info_classif(X, yy, discrete_features=True, random_state=RANDOM_STATE)
rows = []
for sel_name, score_fn in [("chi2", chi2), ("mutual_info", mi_score)]:
    for k in [20, 50, 100, 200, 500, "all"]:
        k_eff = "all" if (k == "all" or k >= total_features) else k
        pipe = Pipeline([("vec", V(**vec_kw)), ("sel", SelectKBest(score_fn, k=k_eff)), ("clf", LogisticRegression(max_iter=2000, C=0.5))])
        sc = cross_val_score(pipe, variants[norm_name], y, cv=cv, scoring="roc_auc")
        rows.append({"selector": sel_name, "k": k, "k_effective": total_features if k_eff == "all" else k_eff, "cv_auc_mean": sc.mean(), "cv_auc_std": sc.std()})
sel = pd.DataFrame(rows); display(sel.round(4))
plt.figure(figsize=(8, 4))
for name, g in sel[sel["k"] != "all"].groupby("selector"): plt.plot(g["k_effective"], g["cv_auc_mean"], marker="o", label=name)
plt.axhline(sel[sel["k"] == "all"]["cv_auc_mean"].iloc[0], color="grey", ls="--", label="no selection"); plt.xscale("log"); plt.legend(); plt.xlabel("k selected terms"); plt.ylabel("CV AUC"); plt.show()

In [ ]:
best_sel = sel.sort_values(["cv_auc_mean"], ascending=False).iloc[0]
# simplest setting within 1 SE of the best (fewer terms = more interpretable)
se = best_sel["cv_auc_std"] / np.sqrt(5)
choice = sel[(sel["cv_auc_mean"] >= best_sel["cv_auc_mean"] - se)].sort_values("k_effective").iloc[0]
print("Best:", dict(best_sel[["selector", "k_effective", "cv_auc_mean"]]), "\nChosen (simplest within 1 SE):", dict(choice[["selector", "k_effective", "cv_auc_mean"]]))

## 5. Final vectorizer, selected terms and their direction

The vectorizer and selector are fitted on all labelled products to list the top terms; a logistic regression gives each term's direction (higher or lower rating). These terms become dataset features.

In [ ]:
vec = V(**vec_kw); Xv = vec.fit_transform(variants[norm_name]); terms = np.array(vec.get_feature_names_out())
chi_s, chi_p = chi2(Xv, y); mi_s = mi_score(Xv, y)
scores = pd.DataFrame({"term": terms, "chi2": chi_s, "chi2_p": chi_p, "mutual_info": mi_s, "doc_count": np.asarray((Xv > 0).sum(axis=0)).ravel()})
key = "chi2" if choice["selector"] == "chi2" else "mutual_info"
scores["rank"] = scores[key].rank(ascending=False, method="first")
top = scores.sort_values(key, ascending=False).head(int(max(N_OUTPUT_TERMS, 1))).copy()
lr = LogisticRegression(max_iter=2000, C=0.5).fit(Xv[:, [list(terms).index(t) for t in top["term"]]], y)
top["direction"] = np.where(lr.coef_[0] > 0, "higher " + target_name.replace("high_", ""), "lower " + target_name.replace("high_", ""))
top["lr_coef"] = lr.coef_[0]
top.round(4)

In [ ]:
# Features for the whole dataset: score every product (including unlabeled ones) with the fitted vectorizer
all_norm = [" ".join(norm_fns[norm_name](tokens(t))) for t in df[title_col]]
Xall = vec.transform(all_norm)
idx = [list(terms).index(t) for t in top["term"]]
safe = [re.sub(r"[^a-z0-9]+", "_", t).strip("_") for t in top["term"]]
tf = pd.DataFrame(Xall[:, idx].toarray(), index=df.index, columns=[f"tf_{s}" for s in safe])
tf["tf_selected_term_count"] = (tf.filter(like="tf_").drop(columns=["tf_selected_term_count"], errors="ignore") > 0).sum(axis=1)
tf["tf_has_any_selected_term"] = (tf["tf_selected_term_count"] > 0).astype(int)
print(tf.shape, "| share of products with at least one selected term:", round(tf["tf_has_any_selected_term"].mean(), 3))

## 6. Save outputs

In [ ]:
out = pd.concat([df.drop(columns=[c for c in tf.columns if c in df.columns]), tf], axis=1)
out_path = PROC / "womens_clothing_tfidf_features.csv"; out.to_csv(out_path, index=False)
exp.round(5).to_csv(REPORTS / "task_20_vectorizer_experiments.csv", index=False)
top.assign(selector=choice["selector"], vectorizer=best_cfg["vectorizer"], ngram=best_cfg["ngram"], stop_words=best_cfg["stop_words"], normalisation=norm_name,
           target=target_name).round(5).to_csv(REPORTS / "task_20_selected_features.csv", index=False)
status = pd.DataFrame([{"task": "Task 20 - TF-IDF / BoW Vectorization", "status": "completed", "run_at": datetime.now().isoformat(timespec="seconds"),
    "input_file": src.name, "rows": len(out), "text_column": title_col, "target": target_name, "products_labelled": int(rel.sum()),
    "best_vectorizer": best_cfg["vectorizer"], "best_ngram": best_cfg["ngram"], "best_stop_words": best_cfg["stop_words"], "best_normalisation": norm_name,
    "vocabulary_size": int(total_features), "selector": choice["selector"], "k_selected": int(choice["k_effective"]),
    "cv_auc_best_config": round(best_cfg["cv_auc_mean"], 4), "cv_auc_selected": round(choice["cv_auc_mean"], 4), "lemmatization_available": lemma_ok,
    "columns_added": out.shape[1] - pd.read_csv(src, nrows=1).shape[1], "output_dataset": out_path.name}])
status.to_csv(REPORTS / "task_20_analysis_status.csv", index=False); status.T

## 7. Key takeaways (fill in after running)

- Which choice moved AUC most: n-grams, stop words, stemming or the vectorizer type? Differences under about 0.01 are within noise.
- Did selection (chi-square or mutual information) keep AUC while cutting the vocabulary from thousands of terms to a few dozen?
- Which terms push ratings (or sales) up or down? These come from short marketing titles, so read them as listing patterns, not causal effects.
- The `tf_*` columns were chosen using the target on all labelled products, so do not reuse them to predict that same target without nesting the selection inside cross-validation (the AUCs above already do).